# CV05 — Mô hình SARIMA(p,d,q)(P,D,Q)[12] & Benchmark đối chuẩn
**Mã công việc:** CV05  
**Phụ trách:** Thành viên 5  

---
## Mô tả Task
Xay dung mo hinh SARIMA voi chu ky mua vu s=12, sai phan mua vu D=1. Chan doan phan du (Ljung-Box tai lag 12, 24). Du bao ngoai mau 41 thang. Lap bang Benchmark doi chuan truc dien voi ARIMA tu CV04. Tra loi cau hoi ly luan 6-9 (Bai 5).

---
## Deliverables (Đầu ra)
- `outputs/models/sarima_model.pkl`
- `outputs/tables/cv05_sarima_metrics.csv`
- `outputs/tables/cv05_model_benchmark_table.csv`
- `outputs/figures/cv05_sarima_forecast_vs_actual.png`
- `outputs/figures/cv05_arima_vs_sarima_comparison.png`

---
## Ràng buộc bắt buộc
1. Chi FIT tren `train.csv` (163 thang), s=12 BAT BUOC.
2. KHONG dung test.csv de chon tham so SARIMA.
3. Du bao Out-of-sample: `get_forecast(steps=41)`.
4. Bang Benchmark doc `cv04_arima_metrics.csv` — KHONG chay lai ARIMA.

---

## 0. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, os
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

ROOT      = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROC_DIR  = os.path.join(ROOT, "data", "processed")
FIG_DIR   = os.path.join(ROOT, "outputs", "figures")
TABLE_DIR = os.path.join(ROOT, "outputs", "tables")
MODEL_DIR = os.path.join(ROOT, "outputs", "models")
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

## 1. Nạp Train & Test

In [ ]:
# train = pd.read_csv(os.path.join(PROC_DIR, "train.csv"),
#                     index_col="date", parse_dates=True)["value"]
# test  = pd.read_csv(os.path.join(PROC_DIR, "test.csv"),
#                     index_col="date", parse_dates=True)["value"]

## 2. Fit mô hình SARIMA(p,d,q)(P,D,Q)[12]

In [ ]:
# Mo hinh de xuat khoi diem cho a10.csv:
#   order          = (1, 1, 1)      — Thanh phan phi mua vu
#   seasonal_order = (0, 1, 1, 12)  — Mua vu 12 thang, D=1
#   (Thu (1,1,1,12) neu PACF gai nhon tai lag 12)

# ORDER          = (1, 1, 1)
# SEASONAL_ORDER = (0, 1, 1, 12)
# model  = SARIMAX(train, order=ORDER, seasonal_order=SEASONAL_ORDER,
#                  enforce_stationarity=False, enforce_invertibility=False)
# result = model.fit(disp=False)
# print(result.summary())
# print(f"AIC: {result.aic:.2f} | BIC: {result.bic:.2f}")

## 3. Chẩn đoán phần dư SARIMA (tập trung lag mùa vụ 12, 24)

In [ ]:
# residuals = result.resid
# lb = acorr_ljungbox(residuals, lags=[12, 24], return_df=True)
# print("Ljung-Box tai lag 12 & 24:")
# print(lb)
# print("=> White Noise tai lag mua vu:", all(lb["lb_pvalue"] > 0.05))

# fig = result.plot_diagnostics(figsize=(14, 8))
# plt.tight_layout()
# plt.show()

## 4. Dự báo ngoài mẫu 41 tháng

In [ ]:
# forecast_obj = result.get_forecast(steps=len(test))
# sarima_fc    = forecast_obj.predicted_mean
# sarima_fc.index = test.index
# conf_int = forecast_obj.conf_int()

## 5. Tính chỉ số đánh giá & Lập bảng Benchmark

In [ ]:
# errors = test.values - sarima_fc.values
# mae  = np.mean(np.abs(errors))
# rmse = np.sqrt(np.mean(errors**2))
# mape = np.mean(np.abs(errors / test.values)) * 100
# print(f"SARIMA MAE={mae:.4f} | RMSE={rmse:.4f} | MAPE={mape:.4f}%")

# --- Lap bang Benchmark ---
# arima_m = pd.read_csv(os.path.join(TABLE_DIR, "cv04_arima_metrics.csv"))
# sarima_row = {"Model": f"SARIMA{ORDER}x{SEASONAL_ORDER}", "AIC": result.aic, "BIC": result.bic,
#              "MAE": mae, "RMSE": rmse, "MAPE": mape}
# benchmark = pd.concat([arima_m, pd.DataFrame([sarima_row])], ignore_index=True)
# benchmark.to_csv(os.path.join(TABLE_DIR, "cv05_model_benchmark_table.csv"), index=False)
# print(benchmark.to_string())

## 6. Vẽ biểu đồ Dự báo SARIMA & So sánh 2 mô hình

In [ ]:
# --- (a) SARIMA forecast vs Actual ---
# fig, ax = plt.subplots(figsize=(14, 5))
# ax.plot(train.index, train, color="steelblue", label="Train")
# ax.plot(test.index, test, color="green", label="Thuc te (Test)")
# ax.plot(sarima_fc.index, sarima_fc, color="red", linestyle="--", label="Du bao SARIMA")
# ax.fill_between(sarima_fc.index, conf_int.iloc[:,0], conf_int.iloc[:,1], alpha=0.2, color="red")
# ax.legend(); ax.set_title("SARIMA — Du bao vs Thuc te")
# fig.savefig(os.path.join(FIG_DIR, "cv05_sarima_forecast_vs_actual.png"), dpi=150, bbox_inches="tight")
# plt.show()

# --- (b) ARIMA vs SARIMA comparison ---
# with open(os.path.join(MODEL_DIR, "arima_model.pkl"), "rb") as f:
#     arima_result = pickle.load(f)
# arima_fc = arima_result.forecast(steps=len(test))
# arima_fc.index = test.index
# fig2, ax2 = plt.subplots(figsize=(14, 5))
# ax2.plot(test.index, test, color="green", label="Thuc te")
# ax2.plot(arima_fc.index,  arima_fc,  color="orange", linestyle="--", label="ARIMA")
# ax2.plot(sarima_fc.index, sarima_fc, color="red",    linestyle="--", label="SARIMA")
# ax2.legend(); ax2.set_title("ARIMA vs SARIMA — Doi chuan tren tap Test")
# fig2.savefig(os.path.join(FIG_DIR, "cv05_arima_vs_sarima_comparison.png"), dpi=150, bbox_inches="tight")
# plt.show()

# with open(os.path.join(MODEL_DIR, "sarima_model.pkl"), "wb") as f:
#     pickle.dump(result, f)
# print("[DONE] CV05 hoan tat.")